# U-Net z próbkowaniem i standaryzacją intensywności

Sprawdzamy dodanie standaryzacji kanałów pacjenta do wariantu z próbkowaniem z notebooka 09. Dla każdego pacjenta obliczamy po jednej średniej i jednym odchyleniu standardowym na kanał z wszystkich jego przekrojów, a następnie wykorzystujemy je podczas odczytu pojedynczych par TIFF.

Przebiegi obejmują seedy 42, 43 i 44, świeżą inicjalizację U-Neta, Adama z `lr=3e-4`, batch size 4, BCE + soft Dice i 40 epok. Rozkład próbkowania, podział pacjentów, próg i kryterium wyboru checkpointu są wspólne z notebookiem 09.

Domyślne `run_training = False` umożliwia przygotowanie danych i przegląd zapisanych wyników. Nowy trening wymaga włączenia flagi oraz własnego `run_name`. Ocenę checkpointów wykonujemy w notebooku 08. Końcowe sekcje zawierają zbiorcze porównanie dotychczasowych eksperymentów i osobną ocenę zapisów ostatniej epoki.

## 1. Biblioteki, ścieżki i konfiguracja

Ścieżki wyznaczamy względem katalogu projektu. Seed steruje inicjalizacją wag i samplera, a `run_name` określa ścieżki wyjściowe. Domyślna konfiguracja opisuje jeden z zakończonych przebiegów.

U-Net zwraca logity `(B, 1, 256, 256)`. Przygotowanie danych zapewnia wejście `float32` o kształcie `(B, 3, 256, 256)`.

In [1]:
from pathlib import Path
import sys
import numpy as np

import torch
import pandas as pd
import tifffile
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import UNet

In [ ]:
run_training = False
run_name = "unet_sampling_zscore_seed44"
seed = 44
learning_rate = 3e-4
batch_size = 4
num_epochs = 40

## 2. Metadane i podział pacjentów

Używamy istniejącego podziału z `splits.csv`. Train ma 77 pacjentów i 2714 przekrojów, a val 16 pacjentów i 615 przekrojów. Zachowujemy również pary z flagą `anomaly_candidate`. Ocena test jest zaplanowana po zakończeniu strojenia i zamrożeniu protokołu.

In [2]:
pairs_df = pd.read_csv(project_root / "data" / "metadata" / "pairs.csv")
splits_df = pd.read_csv(project_root / "data" / "splits" / "splits.csv")
pairs_with_split = pairs_df.merge(
    splits_df, on="patient_id", how="left", validate="many_to_one"
)

val_df = pairs_with_split.loc[pairs_with_split["split"] == "val"].copy()

train_df = pairs_with_split.loc[pairs_with_split["split"] == "train"].copy()
print(f"Train: {len(train_df)} przekrojów, val: {len(val_df)} przekrojów.")

Train: 2714 przekrojów, val: 615 przekrojów.


## 3. Statystyki intensywności dla pacjentów

Łączymy obrazy pacjenta w tablicę `(N, H, W, 3)` uporządkowaną po `slice_id` i dzielimy wartości przez 255. Wywołania `mean(axis=(0, 1, 2))` i `std(axis=(0, 1, 2))` zbierają piksele ze wszystkich przekrojów, pozostawiając po jednej wartości dla każdego kanału. Uwzględniamy także tło i przekroje z pustą maską.

Statystyki liczymy w `float64`. W ćwiczeniu obliczenia w `float32` dawały zauważalny błąd akumulacji, a dokładniejszy wariant odtworzył średnie bliskie 0 i odchylenia bliskie 1 po standaryzacji.

Słowniki `train_stats` i `val_stats` przechowują pary `(m, s)` według `patient_id`. Każdy pacjent korzysta ze statystyk własnego wolumenu. Kontrole sprawdzają liczebności i dodatnie odchylenia. Te odchylenia opisują dane przed standaryzacją.

Port Budy poprzedza standaryzację przeskalowaniem według percentyli. W tym eksperymencie badamy samą standaryzację po dzieleniu przez 255. [Źródłowa normalizacja](https://github.com/mateuszbuda/brain-segmentation-pytorch/blob/master/utils.py#L72-L79).

In [3]:
def compute_patient_stats(patient_df, project_root):
    patient_df = patient_df.sort_values("slice_id")

    volume = np.stack([
        tifffile.imread(project_root / path)
        for path in patient_df["image_path"]
    ]).astype(np.float32) / 255

    m = volume.mean(axis=(0, 1, 2), dtype=np.float64)
    s = volume.std(axis=(0, 1, 2), dtype=np.float64)

    return m, s

In [4]:
train_stats = {}

for patient_id, patient_rows in train_df.groupby("patient_id"):
    m, s = compute_patient_stats(patient_rows, project_root)
    train_stats[patient_id] = (m, s)

print("Liczba pacjentów:", len(train_stats))

stds = np.stack([s for m, s in train_stats.values()])
print("Najmniejsze odchylenie:", stds.min())
print("Wszystkie odchylenia dodatnie:", (stds > 0).all())

Liczba pacjentów: 77
Najmniejsze odchylenie: 0.032733987006888406
Wszystkie odchylenia dodatnie: True


In [5]:
val_stats = {}

for patient_id, patient_rows in val_df.groupby("patient_id"):
    m, s = compute_patient_stats(patient_rows, project_root)
    val_stats[patient_id] = (m, s)

print("Liczba pacjentów:", len(val_stats))

stds = np.stack([s for m, s in val_stats.values()])
print("Najmniejsze odchylenie:", stds.min())
print("Wszystkie odchylenia dodatnie:", (stds > 0).all())

Liczba pacjentów: 16
Najmniejsze odchylenie: 0.034434782555336045
Wszystkie odchylenia dodatnie: True


## 4. Preprocessing przekroju i próbkowanie

Dla odczytanego obrazu wykonujemy `(image / 255 - m) / s` z parametrami jego pacjenta. Broadcasting dopasowuje trzy statystyki do ostatniej osi obrazu. Następnie wynik otrzymuje typ `float32` i układ `(3, H, W)`. Maska ma `float32`, układ `(1, H, W)` i wartości 0/1.

Standaryzowany obraz może mieć wartości ujemne. Średnia bliska 0 i odchylenie bliskie 1 dotyczą całego kanału pacjenta. Poszczególne przekroje zachowują różnice jasności względem siebie.

`MRIDataset` pobiera `m` i `s` ze słownika po `patient_id`. Wagi losowania pozostają takie jak w notebooku 09, wyznaczone z `foreground_pixels` oryginalnych masek. W epoce wykonujemy 2714 losowań ze zwracaniem, czyli 679 aktualizacji przy batch size 4. Łączny budżet wynosi 27 160 aktualizacji.

In [6]:
def preprocess_pair(image, mask, m, s):
    image = image.astype(np.float32) / 255
    image = ((image - m) / s).astype(np.float32)

    image_tensor = torch.from_numpy(image).permute(2, 0, 1)
    mask_tensor = torch.from_numpy(mask > 0).float().unsqueeze(0)

    return image_tensor, mask_tensor

class MRIDataset(Dataset):
    def __init__(self, pairs_df, project_root, patient_stats, transform=None):
        self.pairs_df = pairs_df
        self.project_root = project_root
        self.patient_stats = patient_stats
        self.transform = transform

    def __len__(self):
        return self.pairs_df.shape[0]

    def __getitem__(self, index):
        row = self.pairs_df.iloc[index]
        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]
        m, s = self.patient_stats[row["patient_id"]]
        
        image, mask = preprocess_pair(
            tifffile.imread(image_path),
            tifffile.imread(mask_path),
            m,
            s,
        )

        if self.transform is not None:
            return self.transform(image, mask)
        return image, mask

In [7]:
patient_mean = train_df.groupby("patient_id")["foreground_pixels"].transform("mean")
bonus = patient_mean * 0.1

train_df["sampling_weight"] = train_df["foreground_pixels"] + bonus
train_df["sampling_prob"] = (
    train_df["sampling_weight"] 
    / train_df.groupby("patient_id")["sampling_weight"].transform("sum")
)
train_df.groupby("patient_id")["sampling_prob"].sum().describe()

count    7.700000e+01
mean     1.000000e+00
std      4.932295e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: sampling_prob, dtype: float64

## 5. Funkcja straty, trening i walidacja

Stosujemy BCE dla logitów i soft Dice liczone osobno dla przekrojów, z wagami 1 i 1 oraz `eps=1e-5`. Walidacyjne maski uzyskujemy przy `sigmoid(logits) > 0.5`.

Raportujemy Dice wszystkich przekrojów, przekrojów z adnotacją i pustych masek. Przy pustym wzorcu predykcja pusta otrzymuje 1, a predykcja niepusta 0. Większość przekrojów walidacyjnych ma pustą maskę, dlatego ogólny Dice uzupełniamy pozostałymi metrykami.

In [8]:
def soft_dice_loss(logits, targets, eps=1e-5):
    p = torch.sigmoid(logits)

    num = 2*(p*targets).sum(dim=(1, 2, 3)) + eps
    den = p.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + eps

    return 1 - (num/den).mean()

bce_criterion = torch.nn.BCEWithLogitsLoss()

def bce_dice_loss(logits, targets):
    bce = bce_criterion(logits, targets)
    soft_dice = soft_dice_loss(logits, targets)

    return bce + soft_dice

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    n_samples = 0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, masks)

        loss.backward()
        optimizer.step()

        batch_size = images.shape[0]
        total_loss += loss.item() * batch_size
        n_samples += batch_size

    return total_loss / n_samples

In [10]:
def dice_per_sample(predictions, targets):
    targets_logic = targets > 0

    targets_sum = targets_logic.sum(dim=(1, 2, 3))
    predictions_sum = predictions.sum(dim=(1, 2, 3))
    intersection = (targets_logic & predictions).sum(dim=(1, 2, 3))

    denominator = targets_sum + predictions_sum

    return torch.where(
        denominator == 0,
        1.0,
        2 * intersection / denominator.clamp_min(1)
    )


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    n_samples = 0
    total_dice = 0.0
    total_dice_annotated = 0.0
    total_dice_empty = 0.0
    n_annotated = 0
    n_empty = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            predictions = torch.sigmoid(logits) > 0.5
            dice_scores = dice_per_sample(predictions, masks)
            has_annotation = masks.sum(dim=(1, 2, 3)) > 0

            batch_size = images.shape[0]
            total_loss += loss.item() * batch_size

            n_samples += batch_size
            n_annotated += has_annotation.sum().item()
            n_empty += (~has_annotation).sum().item()

            total_dice += dice_scores.sum().item()
            total_dice_annotated += dice_scores[has_annotation].sum().item()
            total_dice_empty += dice_scores[~has_annotation].sum().item()

    assert n_annotated + n_empty == n_samples

    return {
        "loss": total_loss / n_samples,
        "dice": total_dice / n_samples,
        "dice_annotated": total_dice_annotated / n_annotated,
        "dice_empty": total_dice_empty / n_empty
    }

## 6. Uruchomienie treningu i zapis modelu

Tworzymy świeży U-Net, optymalizator i sampler. Każdy z dwóch zbiorów otrzymuje właściwy słownik statystyk. Checkpoint wybieramy według najwyższego walidacyjnego `dice_annotated`, tak jak w poprzednich przebiegach.

Zapis zawiera wagi, stan optymalizatora, seed, epokę, historię oraz `preprocessing = "patient_channel_zscore"`. Pole to pozwala notebookowi 08 zastosować ten sam preprocessing. Pełna historia trafia do osobnego CSV po 40 epokach.

Dodatkowo zachowano modele ostatniej epoki dla seedów 42 i 44 jako `*_last.pt`. Pliki te zawierają wagi i metadane potrzebne do oceny. Ich wyniki przedstawiamy w osobnym porównaniu diagnostycznym poniżej.

In [17]:
if run_training:
    best_checkpoint_path = (
        project_root / "outputs" / "checkpoints" / f"{run_name}_best.pt"
    )
    history_path = project_root / "outputs" / f"{run_name}_history.csv"
    for path in (best_checkpoint_path, history_path):
        if path.exists():
            raise FileExistsError(f"Wybierz inną nazwę przebiegu. Plik już istnieje: {path}")
    best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    torch.manual_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = UNet().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    train_generator = torch.Generator().manual_seed(seed)

    train_dataset = MRIDataset(train_df, project_root, train_stats)
    val_dataset = MRIDataset(val_df, project_root, val_stats)

    train_sampler = WeightedRandomSampler(
        weights=train_df["sampling_prob"].to_numpy(copy=True),
        num_samples=len(train_dataset),
        replacement=True,
        generator=train_generator,
    )
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, sampler=train_sampler
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    history = []
    best_val_dice = -float("inf")

    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, bce_dice_loss, optimizer, device)
        val_metrics = validate_one_epoch(model, val_loader, bce_dice_loss, device)
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        current_dice = val_metrics["dice_annotated"]
        print(f"Epoka {epoch}/{num_epochs}, train loss = {train_loss:.4f}")
        print("Walidacja", val_metrics)

        if current_dice > best_val_dice:
            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "preprocessing": "patient_channel_zscore",
                    "history": history,
                    "seed": seed,
                },
                best_checkpoint_path,
            )
            best_val_dice = current_dice
            print(f"Zapisano najlepszy model z epoki {epoch}, Dice = {current_dice:.4f}")

    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"Zapisano {len(history)} epok w {history_path.name}")

Epoka 1/40, train loss = 0.4830
Walidacja {'loss': 0.8572468097132396, 'dice': 0.6359031460149501, 'dice_annotated': 0.5342391303787797, 'dice_empty': 0.6992084432717678}
Zapisano najlepszy model z epoki 1, Dice = 0.5342
Epoka 2/40, train loss = 0.3868
Walidacja {'loss': 0.8667953776150215, 'dice': 0.6927248766267202, 'dice_annotated': 0.5001093189595109, 'dice_empty': 0.8126649076517151}
Epoka 3/40, train loss = 0.3761
Walidacja {'loss': 0.8239818624364651, 'dice': 0.5929177472504173, 'dice_annotated': 0.583239044212752, 'dice_empty': 0.5989445910290238}
Zapisano najlepszy model z epoki 3, Dice = 0.5832
Epoka 4/40, train loss = 0.3075
Walidacja {'loss': 0.8060633614295867, 'dice': 0.47416443744810616, 'dice_annotated': 0.6932674951346244, 'dice_empty': 0.33773087071240104}
Zapisano najlepszy model z epoki 4, Dice = 0.6933
Epoka 5/40, train loss = 0.2609
Walidacja {'loss': 0.7844519740440012, 'dice': 0.5028164979888172, 'dice_annotated': 0.7086107880192793, 'dice_empty': 0.374670184696

## 7. Zbiorcze porównanie zakończonych przebiegów

Tabela obejmuje MiniUNet ze stratą BCE i BCE + soft Dice, bazowego U-Neta, próbę większego learning rate, odbicie poziome, próbkowanie oraz próbkowanie ze standaryzacją. Każdy wiersz dotyczy checkpointu wybranego według najwyższego `dice_annotated`.

W MiniUNet budżet wynosił 15 epok i `lr=1e-3`. Przebiegi U-Neta miały 40 epok. Wariant większego learning rate używał `1e-3`, a pozostałe `3e-4`. Wszystkie korzystały z tego samego podziału pacjentów, batch size 4 i progu `> 0.5`. Próby MiniUNet, większego learning rate i odbicia poziomego mają po jednym seedzie. Baza U-Neta, próbkowanie i standaryzacja mają po trzy.

Dla przebiegów z oceną w notebooku 08 metryki pochodzą z `*_val_metrics.csv`. Dla pozostałych odczytujemy metryki przekrojów z historii wybranej epoki. Brakujące wyniki 3D mają wartość `NaN`, ponieważ dla tych przebiegów nie zapisano oceny 3D. Tabela jest zestawieniem historycznym z uwzględnieniem różnych budżetów i konfiguracji.

In [ ]:
metric_columns = {
    "Dice po wszystkich przekrojach": "Dice ogólny",
    "Dice przekrojów z adnotacją": "Dice oznaczonych",
    "Dice przekrojów z pustą maską": "Dice pustych",
    "Dice 3D po pacjentach": "Dice 3D",
    "Dice 3D po pacjentach po filtrowaniu": "Dice 3D po filtrze",
}
summary_rows = []
missing_results = []

for label, filename in (
    ("MiniUNet BCE", "miniunet_bce_best.pt"),
    ("MiniUNet BCE + Dice", "miniunet_bce_dice_best.pt"),
):
    path = project_root / "outputs" / "checkpoints" / filename
    if not path.is_file():
        missing_results.append(filename)
        continue
    checkpoint_summary = torch.load(path, map_location="cpu", weights_only=True)
    best = checkpoint_summary["history"][-1]
    summary_rows.append({
        "wariant": label, "seed": checkpoint_summary["seed"],
        "epoki": 15, "wybrana epoka": checkpoint_summary["epoch"],
        "Dice ogólny": best["dice"],
        "Dice oznaczonych": best["dice_annotated"],
        "Dice pustych": best["dice_empty"],
    })

saved_runs = [
    ("U-Net lr=1e-3", 42, "unet_bce_dice_history_lr1e-3.csv", None),
    ("U-Net baza", 42, "unet_bce_dice_history_lr3e-4.csv", "unet_baseline_seed42"),
    ("U-Net baza", 43, "unet_bce_dice_best_lr3e-4_seed43.csv", "unet_baseline_seed43"),
    ("U-Net baza", 44, "unet_bce_dice_best_lr3e-4_seed44.csv", "unet_baseline_seed44"),
    ("U-Net odbicie p=0.5", 42, "unet_bce_dice_best_lr3e-4_hflip.csv", None),
]
for run_seed in (42, 43, 44):
    prefix = f"unet_buda_sampling_seed{run_seed}"
    saved_runs.append(("U-Net próbkowanie", run_seed, f"{prefix}_history.csv", prefix))
for run_seed in (42, 43, 44):
    prefix = f"unet_sampling_zscore_seed{run_seed}"
    saved_runs.append(("U-Net próbkowanie + z-score", run_seed, f"{prefix}_history.csv", prefix))

for label, run_seed, history_filename, metrics_prefix in saved_runs:
    history_path = project_root / "outputs" / history_filename
    if not history_path.is_file():
        missing_results.append(history_filename)
        continue
    run_history = pd.read_csv(history_path)
    best = run_history.loc[run_history["dice_annotated"].idxmax()]
    row = {
        "wariant": label, "seed": run_seed,
        "epoki": len(run_history), "wybrana epoka": int(best["epoch"]),
        "Dice ogólny": best["dice"],
        "Dice oznaczonych": best["dice_annotated"],
        "Dice pustych": best["dice_empty"],
    }
    if metrics_prefix is not None:
        metrics_path = project_root / "outputs" / f"{metrics_prefix}_val_metrics.csv"
        if metrics_path.is_file():
            saved_metrics = pd.read_csv(metrics_path, index_col=0).iloc[:, 0]
            row.update({column: saved_metrics[key] for key, column in metric_columns.items()})
        else:
            missing_results.append(metrics_path.name)
    summary_rows.append(row)

if missing_results:
    print("Brak lokalnych plików:", ", ".join(missing_results))

experiment_summary = pd.DataFrame(summary_rows)
experiment_summary.round(4)

,wariant,seed,epoki,wybrana epoka,Dice ogólny,Dice oznaczonych,Dice pustych,Dice 3D,Dice 3D po filtrze
0,MiniUNet BCE,42,15,14,0.6243,0.2455,0.8602,NaN,NaN
1,MiniUNet BCE + Dice,42,15,9,0.4783,0.3353,0.5673,NaN,NaN
2,U-Net lr=1e-3,42,40,2,0.4972,0.3380,0.5963,NaN,NaN
3,U-Net baza,42,40,6,0.4799,0.4625,0.4908,0.4680,0.5102
4,U-Net baza,43,40,15,0.5221,0.4835,0.5462,0.5034,0.5626
5,U-Net baza,44,40,7,0.5165,0.4688,0.5462,0.4345,0.5200
6,U-Net odbicie p=0.5,42,40,4,0.5638,0.3675,0.6860,NaN,NaN
7,U-Net próbkowanie,42,40,36,0.7505,0.6084,0.8391,0.6561,0.6651
8,U-Net próbkowanie,43,40,27,0.5684,0.6380,0.5251,0.6445,0.6775
9,U-Net próbkowanie,44,40,18,0.5488,0.7224,0.4406,0.6756,0.7589


## 8. Porównanie trzech seedów przy wspólnym budżecie

Baza U-Neta, próbkowanie oraz próbkowanie ze standaryzacją mają po trzy przebiegi z seedami 42, 43 i 44. Wspólne są model, strata, Adam z `lr=3e-4`, batch size 4, 40 epok, podział pacjentów i reguła wyboru checkpointu.

Tabela przedstawia średnią ± odchylenie standardowe z próby między seedami, z `ddof=1`. Każdy wynik 3D jest wcześniej średnią po tych samych 16 pacjentach val. Odchylenie opisuje zmienność treningu na tym podziale.

In [ ]:
aggregate_rows = []
for label in ("U-Net baza", "U-Net próbkowanie", "U-Net próbkowanie + z-score"):
    group = experiment_summary.loc[experiment_summary["wariant"] == label]
    if set(group["seed"]) != {42, 43, 44} or len(group) != 3:
        print(f"Niepełne trzy powtórzenia dla: {label}")
        continue
    row = {"wariant": label, "liczba seedów": len(group)}
    for column in ("Dice ogólny", "Dice oznaczonych", "Dice pustych", "Dice 3D", "Dice 3D po filtrze"):
        values = group[column]
        row[column] = f"{values.mean():.4f} ± {values.std(ddof=1):.4f}"
    aggregate_rows.append(row)

series_summary = pd.DataFrame(aggregate_rows)
series_summary

,wariant,liczba seedów,Dice ogólny,Dice oznaczonych,Dice pustych,Dice 3D,Dice 3D po filtrze
0,U-Net baza,3,0.5062 ± 0.0229,0.4716 ± 0.0108,0.5277 ± 0.0320,0.4686 ± 0.0344,0.5309 ± 0.0279
1,U-Net próbkowanie,3,0.6226 ± 0.1113,0.6563 ± 0.0592,0.6016 ± 0.2099,0.6587 ± 0.0157,0.7005 ± 0.0510
2,U-Net próbkowanie + z-score,3,0.5973 ± 0.0516,0.7558 ± 0.0106,0.4987 ± 0.0881,0.6750 ± 0.0099,0.7533 ± 0.0241


## 9. Dodatkowa ocena ostatniej epoki

Dla standaryzacji zapisano dodatkowo ostatnią epokę dla seedów 42 i 44. Te modele oceniamy na tym samym val, tym samym progu i z tym samym filtrowaniem. Główne porównanie trzech seedów powyżej korzysta z checkpointów wybranych według `dice_annotated`.

Porównanie ostatniej epoki jest analizą dodatkową wykonaną po przejrzeniu historii. Pozwala zobaczyć zależność wyniku od kryterium wyboru. Dla epok 31 i 33 przebiegu seeda 42 zachowano metryki w historii. Ocena 3D tych epok wymagałaby odpowiadających im wag.

In [ ]:
last_epoch_rows = []
for run_seed in (42, 44):
    prefix = f"unet_sampling_zscore_seed{run_seed}"
    history_path = project_root / "outputs" / f"{prefix}_history.csv"
    if not history_path.is_file():
        print(f"Brak historii dla: {prefix}")
        continue
    run_history = pd.read_csv(history_path)
    best_epoch = int(run_history.loc[run_history["dice_annotated"].idxmax(), "epoch"])
    last_epoch = int(run_history.iloc[-1]["epoch"])
    for suffix, selection, epoch_number in (
        ("", "najwyższy dice_annotated", best_epoch),
        ("_last", "ostatnia epoka", last_epoch),
    ):
        metrics_path = project_root / "outputs" / f"{prefix}{suffix}_val_metrics.csv"
        if not metrics_path.is_file():
            print(f"Brak oceny: {metrics_path.name}")
            continue
        saved_metrics = pd.read_csv(metrics_path, index_col=0).iloc[:, 0]
        row = {"seed": run_seed, "wybór": selection, "epoka": epoch_number}
        row.update({column: saved_metrics[key] for key, column in metric_columns.items()})
        last_epoch_rows.append(row)

last_epoch_summary = pd.DataFrame(last_epoch_rows)
last_epoch_summary.round(4)

,seed,wybór,epoka,Dice ogólny,Dice oznaczonych,Dice pustych,Dice 3D,Dice 3D po filtrze
0,42,najwyższy dice_annotated,27,0.5915,0.7659,0.4828,0.6856,0.7761
1,42,ostatnia epoka,40,0.7325,0.7436,0.7256,0.7030,0.7550
2,44,najwyższy dice_annotated,31,0.6516,0.7447,0.5937,0.6735,0.7555
3,44,ostatnia epoka,40,0.7777,0.7045,0.8232,0.6916,0.7119


## Wnioski i dalsza praca

Dodanie standaryzacji zwiększyło średni Dice oznaczonych przekrojów z 0,6563 ± 0,0592 do 0,7558 ± 0,0106. Średni Dice 3D wzrósł z 0,6587 ± 0,0157 do 0,6750 ± 0,0099, a po filtrowaniu z 0,7005 ± 0,0510 do 0,7533 ± 0,0241.

Poprawę 3D zaobserwowano dla seedów 42 i 43. Dla seeda 44 wynik 3D był minimalnie niższy niż w wariancie z samym próbkowaniem. Średni Dice pustych masek zmniejszył się z 0,6016 do 0,4987, co wskazuje na częstsze niepuste predykcje przy pustym wzorcu.

Ostatnia epoka dla seedów 42 i 44 miała wyższy Dice 3D przed filtrowaniem oraz niższy Dice 3D po filtrowaniu niż wybrane checkpointy. Wybór kryterium należy powiązać z docelowym sposobem oceny całej predykcji.

Kolejny etap, zaplanowany na następny chat, obejmuje wydzielenie wspólnego kodu do `src` i automatyzację konfiguracji, seedów, nazw plików, oceny oraz raportów. Propozycja dla nowej serii obejmuje wybór checkpointu według walidacyjnego Dice 3D przed filtrowaniem i automatyczny zapis ostatniej epoki. Wspólny protokół nowej serii wymaga ustalenia przed jej uruchomieniem i objęcia nim wariantu kontrolnego.

Pierwszym planowanym modułem jest `src/preprocessing.py`. Ma obsługiwać przygotowanie parametrów pacjenta i ich zastosowanie do przekroju. Kolejnym kandydatem do badania jest przeskalowanie według percentyli przed standaryzacją. Plan obejmuje też osobne próby BatchNorm i funkcji straty, augmentacje oraz ewentualnie mocniejszą architekturę. Parametry i kolejność kolejnych eksperymentów pozostają do ustalenia.